In [1]:
# ==============================================================================
# === STAGE 1: TRAIN ON KAGGLE MRI DATASET WITH LORA (PYTORCH) ===
# ==============================================================================

# --- 1.1. IMPORTS AND SETUP ---
import os
import time
import shutil
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, TensorDataset
from torchvision import transforms
from transformers import ViTModel
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils import class_weight
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import cv2

# --- 1.2. CONSTANTS ---
IMG_SIZE = 224
BATCH_SIZE = 16
KAGGLE_EPOCHS = 30
OASIS_EPOCHS = 50

# --- 1.3. DEVICE CONFIGURATION ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# --- 1.4. KAGGLE DATA PREPARATION ---
print("\n--- Preparing Kaggle Dataset ---")
# Note: Adjusted paths for a generic environment.
# Make sure your data is in '/kaggle/input/research-dataset-v2/Kaggle' or adjust as needed.
source_base_dir = "/kaggle/input/research-dataset-v2/Kaggle"
dest_base_dir = "cleaned/Kaggle"
if os.path.exists(dest_base_dir):
    shutil.rmtree(dest_base_dir)

if os.path.exists(source_base_dir):
    for split in ["Training", "Testing"]:
        source_dir = os.path.join(source_base_dir, split)
        dest_dir = os.path.join(dest_base_dir, split)
        if not os.path.exists(source_dir): continue
        for class_name in tqdm(os.listdir(source_dir), desc=f"Copying Kaggle {split} data"):
            source_class_dir = os.path.join(source_dir, class_name)
            dest_class_dir = os.path.join(dest_dir, class_name)
            if not os.path.isdir(source_class_dir): continue
            os.makedirs(dest_class_dir, exist_ok=True)
            for img_file in os.listdir(source_class_dir):
                shutil.copy(os.path.join(source_class_dir, img_file), os.path.join(dest_class_dir, img_file))

class ImageDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        img_path = self.df.iloc[idx]["image_path"]
        label = self.df.iloc[idx]["label"]
        image = cv2.imread(img_path)
        if image is None: return torch.zeros((3, IMG_SIZE, IMG_SIZE)), -1
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        if self.transform: image = self.transform(image)
        return image, label

def create_df_from_dir(directory):
    all_files = []
    if not os.path.exists(directory): return pd.DataFrame(all_files)
    for class_name in os.listdir(directory):
        class_dir = os.path.join(directory, class_name)
        if os.path.isdir(class_dir):
            for img_file in os.listdir(class_dir):
                all_files.append({"image_path": os.path.join(class_dir, img_file), "label_name": class_name})
    return pd.DataFrame(all_files)

kaggle_df_train = create_df_from_dir("cleaned/Kaggle/Training")
kaggle_df_val = create_df_from_dir("cleaned/Kaggle/Testing")

# Handle case where Kaggle data might not be present
if not kaggle_df_train.empty and not kaggle_df_val.empty:
    label_to_idx_kaggle = {label: idx for idx, label in enumerate(sorted(kaggle_df_train["label_name"].unique()))}
    kaggle_df_train["label"] = kaggle_df_train["label_name"].map(label_to_idx_kaggle)
    kaggle_df_val["label"] = kaggle_df_val["label_name"].map(label_to_idx_kaggle)
    NUM_CLASSES_KAGGLE = len(label_to_idx_kaggle)

    transform_kaggle = transforms.Compose([transforms.ToPILImage(), transforms.Resize((IMG_SIZE, IMG_SIZE)), transforms.ToTensor(), transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])])
    kaggle_train_loader = DataLoader(ImageDataset(df=kaggle_df_train, transform=transform_kaggle), batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    kaggle_val_loader = DataLoader(ImageDataset(df=kaggle_df_val, transform=transform_kaggle), batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    print(f"Kaggle DataLoaders Ready: Train = {len(kaggle_train_loader.dataset)} samples, Val = {len(kaggle_val_loader.dataset)} samples")

# --- 1.5. FEATURE EXTRACTION ---
print("\n--- Extracting Features from Kaggle Dataset ---")
vit = ViTModel.from_pretrained("google/vit-base-patch16-224-in21k").to(device)
vit.eval()

def extract_cls_tokens(dataloader, desc):
    cls_tokens_list, label_list = [], []
    with torch.no_grad():
        for images, labels in tqdm(dataloader, desc=desc):
            valid_indices = labels != -1
            if not valid_indices.any(): continue
            images, labels = images[valid_indices], labels[valid_indices]
            outputs = vit(pixel_values=images.to(device))
            cls_tokens_list.append(outputs.last_hidden_state[:, 0, :].cpu())
            label_list.append(labels)
    return torch.cat(cls_tokens_list), torch.cat(label_list)

if not kaggle_df_train.empty and not kaggle_df_val.empty:
    train_cls_tokens_k, train_labels_k = extract_cls_tokens(kaggle_train_loader, "Extracting Kaggle Train CLS Tokens")
    val_cls_tokens_k, val_labels_k = extract_cls_tokens(kaggle_val_loader, "Extracting Kaggle Val CLS Tokens")
    train_loader_cached_k = DataLoader(TensorDataset(train_cls_tokens_k, train_labels_k), batch_size=BATCH_SIZE, shuffle=True)
    val_loader_cached_k = DataLoader(TensorDataset(val_cls_tokens_k, val_labels_k), batch_size=BATCH_SIZE, shuffle=False)

# --- 1.6. MODEL DEFINITION & TRAINING FUNCTION (LoRA) ---
# class LoRAAdapter(nn.Module):
#     def __init__(self, dim, rank=4, alpha=8):
#         super().__init__()
#         self.A = nn.Linear(dim, rank, bias=False)
#         self.B = nn.Linear(rank, dim, bias=False)
#         self.scale = alpha / rank
#         nn.init.zeros_(self.B.weight)
#     def forward(self, x):
#         return self.B(self.A(x)) * self.scale

# class LoRAClassifier(nn.Module):
#     def __init__(self, input_dim=768, num_classes=4, rank=4, alpha=8):
#         super().__init__()
#         self.lora = LoRAAdapter(input_dim, rank, alpha)
#         self.classifier = nn.Sequential(nn.Linear(input_dim, 256), nn.ReLU(), nn.Dropout(0.3), nn.Linear(256, num_classes))
#     def forward(self, x):
#         x = x + self.lora(x)
#         return self.classifier(x)


class LoRAAdapter(nn.Module):
    def __init__(self, dim, rank=8, alpha=16): # <-- TUNE HERE (e.g., rank=8, alpha=16)
        super().__init__()
        self.A = nn.Linear(dim, rank, bias=False)
        self.B = nn.Linear(rank, dim, bias=False)
        self.scale = alpha / rank
        nn.init.zeros_(self.B.weight)
    def forward(self, x):
        return self.B(self.A(x)) * self.scale

class LoRAClassifier(nn.Module):
    def __init__(self, input_dim=768, num_classes=4, rank=8, alpha=16): # <-- TUNE HERE
        super().__init__()
        self.lora = LoRAAdapter(input_dim, rank, alpha)
        # You can also adjust the dropout rate here
        self.classifier = nn.Sequential(nn.Linear(input_dim, 256), nn.ReLU(), nn.Dropout(0.4), nn.Linear(256, num_classes)) # <-- TUNE DROPOUT
    def forward(self, x):
        x = x + self.lora(x)
        return self.classifier(x)

# ==============================================================================
# === THIS FUNCTION CONTAINS THE FIX ===
# ==============================================================================
def train_model(model, train_loader, val_loader, optimizer, criterion, num_epochs, model_save_path):
    best_val_accuracy = 0.0
    for epoch in range(num_epochs):
        model.train()
        for feats, labels in train_loader:
            feats, labels = feats.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(feats)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
        
        model.eval()
        val_correct, val_total = 0, 0
        with torch.no_grad():
            for feats, labels in val_loader:
                outputs = model(feats.to(device))
                _, preds = torch.max(outputs, 1)
                val_total += labels.size(0)
                val_correct += (preds == labels.to(device)).sum().item()
        val_acc = val_correct / val_total
        print(f"Epoch {epoch+1}/{num_epochs} | Val Acc: {val_acc:.4f}")

        if val_acc > best_val_accuracy:
            best_val_accuracy = val_acc
            # **FIXED LINE:** Save the state_dict of the lora submodule directly.
            # This is cleaner and avoids the "lora." prefix mismatch later.
            torch.save(model.lora.state_dict(), model_save_path)
            print(f"✔️ Saved best LoRA weights with Val Acc: {val_acc:.4f}")

# --- 1.7. EXECUTE STAGE 1 TRAINING ---
if not kaggle_df_train.empty and not kaggle_df_val.empty:
    print("\n--- Starting Stage 1: Kaggle Model Training ---")
    model_kaggle = LoRAClassifier(num_classes=NUM_CLASSES_KAGGLE).to(device)
    optimizer_kaggle = torch.optim.Adam(model_kaggle.parameters(), lr=1e-3)
    criterion_kaggle = nn.CrossEntropyLoss()
    train_model(model_kaggle, train_loader_cached_k, val_loader_cached_k, optimizer_kaggle, criterion_kaggle, num_epochs=KAGGLE_EPOCHS, model_save_path="best_lora_only_kaggle.pth")
    print("--- Stage 1 Training Finished ---")
else:
    print("\n--- Skipping Stage 1: Kaggle data not found ---")


# ========================================================================================
# === STAGE 2: TRANSFER LORA (FROZEN) + TUNE ADAPTER PROMPT ON OASIS DATASET (PYTORCH) ===
# ========================================================================================

# --- 2.1. OASIS DATA PREPARATION ---
print("\n\n--- Preparing OASIS Dataset ---")
OASIS_ROOT = "/kaggle/input/research-dataset-v2/OASIS/OASIS"
oasis_df = create_df_from_dir(OASIS_ROOT)

if not oasis_df.empty:
    class_map_o = {"Non Demented": 0, "Demented": 1}
    oasis_df['label'] = oasis_df['label_name'].map(class_map_o)
    oasis_train_df, oasis_val_df = train_test_split(oasis_df, test_size=0.3, stratify=oasis_df["label"], random_state=42)
    NUM_CLASSES_OASIS = 2

    # transform_oasis = transforms.Compose([transforms.ToPILImage(), transforms.Resize((IMG_SIZE, IMG_SIZE)), transforms.ToTensor(), transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])])
    # oasis_train_loader = DataLoader(ImageDataset(df=oasis_train_df, transform=transform_oasis), batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    # oasis_val_loader = DataLoader(ImageDataset(df=oasis_val_df, transform=transform_oasis), batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    # print(f"OASIS DataLoaders Ready: Train = {len(oasis_train_loader.dataset)} samples, Val = {len(oasis_val_loader.dataset)} samples")

    # --- 2.1. OASIS DATA PREPARATION ---
    
    transform_oasis_train = transforms.Compose([
        transforms.ToPILImage(),
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(10), # Adds slight rotation
        transforms.ColorJitter(brightness=0.1, contrast=0.1), # Adjusts brightness/contrast
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    # IMPORTANT: No augmentation on the validation set
    transform_oasis_val = transforms.Compose([
        transforms.ToPILImage(),
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    oasis_train_loader = DataLoader(ImageDataset(df=oasis_train_df, transform=transform_oasis_train), batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    oasis_val_loader = DataLoader(ImageDataset(df=oasis_val_df, transform=transform_oasis_val), batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    print(f"OASIS DataLoaders Ready: Train = {len(oasis_train_loader.dataset)} samples, Val = {len(oasis_val_loader.dataset)} samples")

    # --- 2.2. OASIS FEATURE EXTRACTION ---
    print("\n--- Extracting Features from OASIS Dataset ---")
    train_cls_tokens_o, train_labels_o = extract_cls_tokens(oasis_train_loader, "Extracting OASIS Train CLS Tokens")
    val_cls_tokens_o, val_labels_o = extract_cls_tokens(oasis_val_loader, "Extracting OASIS Val CLS Tokens")
    train_loader_cached_o = DataLoader(TensorDataset(train_cls_tokens_o, train_labels_o), batch_size=BATCH_SIZE, shuffle=True)
    val_loader_cached_o = DataLoader(TensorDataset(val_cls_tokens_o, val_labels_o), batch_size=BATCH_SIZE, shuffle=False)

    # --- 2.3. NEW HYBRID MODEL DEFINITION (LoRA + VPT) ---
    # class LoRA_VPT_Classifier(nn.Module):
    #     def __init__(self, input_dim=768, num_classes=2, rank=4, alpha=8):
    #         super().__init__()
    #         self.lora = LoRAAdapter(input_dim, rank, alpha)
    #         self.prompt = nn.Parameter(torch.zeros(1, input_dim))
    #         self.classifier = nn.Sequential(nn.Linear(input_dim, 256), nn.ReLU(), nn.Dropout(0.3), nn.Linear(256, num_classes))

    #     def forward(self, x):
    #         lora_x = x + self.lora(x)
    #         prompted_x = lora_x + self.prompt
    #         return self.classifier(prompted_x)

    class LoRA_VPT_Classifier(nn.Module):
        def __init__(self, input_dim=768, num_classes=2, rank=8, alpha=16): # <-- TUNE HERE
            super().__init__()
            self.lora = LoRAAdapter(input_dim, rank, alpha)
            self.prompt = nn.Parameter(torch.zeros(1, input_dim))
            # And here as well
            self.classifier = nn.Sequential(nn.Linear(input_dim, 256), nn.ReLU(), nn.Dropout(0.4), nn.Linear(256, num_classes)) # <-- TUNE DROPOUT
    
        def forward(self, x):
            lora_x = x + self.lora(x)
            prompted_x = lora_x + self.prompt
            return self.classifier(prompted_x)

    # --- 2.4. BUILD OASIS MODEL, LOAD & FREEZE LORA WEIGHTS ---
    print("\n--- Building Stage 2 Hybrid Model for OASIS ---")
    model_oasis_hybrid = LoRA_VPT_Classifier(num_classes=NUM_CLASSES_OASIS).to(device)

    # The loading code now works without modification because the saving method was fixed.
    if os.path.exists("best_lora_only_kaggle.pth"):
        print("Loading pre-trained LoRA weights from Stage 1...")
        lora_weights_from_kaggle = torch.load("best_lora_only_kaggle.pth")
        model_oasis_hybrid.lora.load_state_dict(lora_weights_from_kaggle)
        print("LoRA weights successfully loaded!")

        for param in model_oasis_hybrid.lora.parameters():
            param.requires_grad = False
        print("LoRA adapter has been frozen. Only the prompt and classifier will be trained.")
    else:
        print("WARNING: Pre-trained LoRA weights not found. Training from scratch.")

    # --- 2.5. HANDLE CLASS IMBALANCE ---
    print("\n--- Calculating Class Weights for OASIS ---")
    oasis_class_weights = class_weight.compute_class_weight('balanced', classes=np.unique(oasis_train_df['label']), y=oasis_train_df['label'])
    oasis_class_weights = torch.tensor(oasis_class_weights, dtype=torch.float).to(device)
    print(f"OASIS Class Weights: {oasis_class_weights.cpu().numpy()}")

    # --- 2.6. EXECUTE STAGE 2 TRAINING (HYBRID METHOD) ---
    def train_hybrid_model(model, train_loader, val_loader, optimizer, criterion, num_epochs, model_save_path):
        best_val_accuracy = 0.0
        for epoch in range(num_epochs):
            model.train()
            for feats, labels in train_loader:
                feats, labels = feats.to(device), labels.to(device)
                optimizer.zero_grad()
                outputs = model(feats)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()
            
            model.eval()
            val_correct, val_total = 0, 0
            with torch.no_grad():
                for feats, labels in val_loader:
                    outputs = model(feats.to(device))
                    _, preds = torch.max(outputs, 1)
                    val_total += labels.size(0)
                    val_correct += (preds == labels.to(device)).sum().item()
            
            val_acc = val_correct / val_total
            print(f"Epoch {epoch+1}/{num_epochs} | Val Acc: {val_acc:.4f}")
            
            if val_acc > best_val_accuracy:
                best_val_accuracy = val_acc
                torch.save(model.state_dict(), model_save_path)
                print(f"✔️ Saved best HYBRID model with Val Acc: {val_acc:.4f}")

    print("\n--- Starting Stage 2: OASIS Model Fine-Tuning (Prompt Only) ---")
    # Filter parameters to only train the prompt and classifier
    optimizer_oasis = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model_oasis_hybrid.parameters()),
        lr=1e-4
    )
    criterion_oasis = nn.CrossEntropyLoss(weight=oasis_class_weights)

    train_hybrid_model(model_oasis_hybrid, train_loader_cached_o, val_loader_cached_o, optimizer_oasis, criterion_oasis, num_epochs=OASIS_EPOCHS, model_save_path="best_hybrid_oasis_model.pth")
    print("--- Stage 2 Training Finished ---")

    # --- 2.7. FINAL EVALUATION ON OASIS DATASET ---
    print("\n\n--- Final Evaluation on OASIS Validation Set ---")
    model_oasis_hybrid.load_state_dict(torch.load("best_hybrid_oasis_model.pth"))
    model_oasis_hybrid.eval()

    all_preds_o, all_labels_o = [], []
    with torch.no_grad():
        for feats, labels in val_loader_cached_o:
            outputs = model_oasis_hybrid(feats.to(device)); _, preds = torch.max(outputs, 1)
            all_preds_o.extend(preds.cpu().numpy()); all_labels_o.extend(labels.cpu().numpy())

    oasis_class_names = ["Non Demented", "Demented"]
    print("=== OASIS Classification Report (Hybrid Method) ===")
    print(classification_report(all_labels_o, all_preds_o, target_names=oasis_class_names))

    cm = confusion_matrix(all_labels_o, all_preds_o)
    plt.figure(figsize=(8, 6)); sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=oasis_class_names, yticklabels=oasis_class_names)
    plt.title("Confusion Matrix (OASIS - Hybrid Method)"); plt.xlabel("Predicted Labels"); plt.ylabel("True Labels"); plt.show()
else:
    print("\n--- Skipping Stage 2: OASIS data not found ---")

SyntaxError: invalid non-printable character U+00A0 (2575694394.py, line 270)